# Notebook 08 – Personality Nudges

Generate **lightweight personality refinements** for styling adjustments.

Personality **never replaces the interface** — it only nudges configurable properties such as:

- Visual Richness
- Information Density
- Whitespace
- Animation
- Recommendation Strength

For every Big Five trait level (Low / Medium / High), survey tendencies supported by Notebook 03 and Notebook 04 evidence are converted into ordinal nudges (`-1`, `+1`).


## Inputs
- `data/processed/clean_dataset.csv`
- Notebook 03 statistical results
- Notebook 04 Random Forest + SHAP

## Outputs
- `data/outputs/trait_modifiers.json`
- `reports/PersonalityNudges/trait_modifiers.xlsx`


In [1]:
import json
import logging
import sys
from pathlib import Path

import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate
    raise FileNotFoundError("Could not find project root containing src/config.py.")


PROJECT_ROOT = _bootstrap_project()

from src.personality_nudges.repository import run_personality_nudge_pipeline
from src.utils.notebook import setup_notebook

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PATHS, REPORTS = setup_notebook("PersonalityNudges")
OUTPUT_DIR = PATHS.data_outputs

print(f"Reports: {REPORTS}")
print(f"JSON output: {OUTPUT_DIR}")


Reports: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/PersonalityNudges
JSON output: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs


## Check Inputs


In [2]:
INPUTS = {
    "Clean dataset": PATHS.data_processed / "clean_dataset.csv",
    "Statistical results (NB03)": PATHS.reports / "Statistical_Validation" / "tables" / "statistical_results.xlsx",
    "Feature importance (NB04)": PATHS.reports / "Feature_Importance" / "feature_importance.xlsx",
    "SHAP summary (NB04)": PATHS.reports / "Feature_Importance" / "shap_summary.csv",
}

for label, path in INPUTS.items():
    print(f"{label}: {'OK' if path.exists() else 'MISSING'}")


Clean dataset: OK
Statistical results (NB03): OK
Feature importance (NB04): OK
SHAP summary (NB04): OK


## Generate Evidence-Backed Personality Nudges

1. Derive ordinal tendencies from the survey for each trait × level × property
2. Score with statistical + ML evidence (Notebook 03–04)
3. Export only nudges meeting the evidence threshold


In [3]:
result = run_personality_nudge_pipeline(PROJECT_ROOT, OUTPUT_DIR, REPORTS)

entries = result.entries
scored = result.scored_modifiers
summary = result.summary

display(scored.head(15) if not scored.empty else scored)


INFO: Personality nudges: 9 entries, 14 total nudges


,Trait,Level,Property,Nudge,Direction,Delta,Group_N,Baseline_Score,Group_Score,Provenance,Cramers_V,RF_Importance,Mean_SHAP,Effect_Size,Coverage,Modifier_Evidence_Score,Strength
0,Agreeableness,High,recommendation_emphasis,1,increase,NaN,99,0.7232,0.7197,theory,0.097985,0.122641,0.025799,0.0000,0.495,0.306610,Moderate
1,Conscientiousness,High,information_density,1,increase,NaN,68,0.5312,0.5221,theory,0.109735,0.138486,0.026730,0.0000,0.340,0.319887,Moderate
2,Conscientiousness,Low,whitespace,-1,decrease,-0.0570,43,0.5454,0.4884,data-driven,0.174924,0.163429,0.037647,0.0570,0.215,0.807238,Very Strong
3,Conscientiousness,Low,information_density,-1,decrease,NaN,43,0.5312,0.5465,theory,0.109735,0.138486,0.026730,0.0000,0.215,0.308637,Moderate
4,Extraversion,High,recommendation_emphasis,-1,decrease,-0.0793,33,0.7232,0.6439,data-driven,0.134106,0.127419,0.018839,0.0793,0.165,0.649328,Strong
5,Extraversion,High,information_density,-1,decrease,-0.0616,33,0.5312,0.4697,data-driven,0.106715,0.137781,0.023054,0.0616,0.165,0.602851,Strong
6,Extraversion,High,animation_level,1,increase,NaN,33,NaN,NaN,theory,0.000000,0.000000,0.000000,0.0000,0.165,0.014850,Weak
7,Extraversion,Low,recommendation_emphasis,-1,decrease,NaN,64,0.7232,0.7031,theory,0.134106,0.127419,0.018839,0.0000,0.320,0.313547,Moderate
8,Neuroticism,High,recommendation_emphasis,1,increase,0.0528,48,0.7232,0.7760,data-driven,0.179644,0.151059,0.037582,0.0528,0.240,0.792847,Very Strong
9,Neuroticism,High,information_density,-1,decrease,NaN,48,0.5312,0.5573,theory,0.122399,0.175576,0.032592,0.0000,0.240,0.368946,Moderate


## Sample Nudge JSON


In [4]:
if entries:
    print(json.dumps(entries[0], indent=2, ensure_ascii=False))


{
  "trait": "Extraversion",
  "level": "Low",
  "nudges": {
    "recommendation_strength": -1
  },
  "confidence": 0.313547
}


## Modifiers per Trait


In [5]:
for trait, count in summary["modifiers_per_trait"].items():
    print(f"{trait}: {count}")


Extraversion: 3
Agreeableness: 1
Conscientiousness: 3
Neuroticism: 4
Openness: 3


## Exports


In [6]:
print("Export locations:")
for name, path in result.export_paths.items():
    print(f"- {name}: {path}")


Export locations:
- trait_modifiers_json: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs/trait_modifiers.json
- trait_modifiers_xlsx: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/PersonalityNudges/trait_modifiers.xlsx
- trait_modifiers_csv: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/PersonalityNudges/trait_modifiers.csv
- summary_md: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/PersonalityNudges/personality_nudges_summary.md


## Final Output


In [7]:
print("Modifiers per trait")
for trait, count in summary["modifiers_per_trait"].items():
    print(f"  {trait}: {count}")
print("Repository generated.")


Modifiers per trait
  Extraversion: 3
  Agreeableness: 1
  Conscientiousness: 3
  Neuroticism: 4
  Openness: 3
Repository generated.
